In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

In [2]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [6]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [7]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir / "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [8]:
patch_size = 15

train_dataset = SalinasPatchDataset(
    scene,
    indices=train_indices,
    patch_size=patch_size,
)

val_dataset = SalinasPatchDataset(
    scene,
    indices=val_indices,
    patch_size=patch_size,
)

test_dataset = SalinasPatchDataset(
    scene,
    indices=test_indices,
    patch_size=patch_size,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

print("DataLoaders created.")

DataLoaders created.


In [10]:
sample_x, sample_y = train_dataset[0]

print("Sample patch:", sample_x.shape)
print("Sample label:", sample_y)

Sample patch: (204, 15, 15)
Sample label: 7


In [11]:
from dfm.models.hybrid import SpectralAttentionBranch

In [12]:
class SpectralOnlyClassifier(nn.Module):
    """
    Spectral-only ablation.

    Uses the exact SpectralAttentionBranch from the Hybrid model,
    followed by the same classification head.
    """

    def __init__(
        self,
        num_classes,
        feature_dim=128,
        spectral_depth=2,
        spectral_heads=4,
        dropout=0.1,
        max_bands=256,
    ):
        super().__init__()

        self.spectral = SpectralAttentionBranch(
            feature_dim=feature_dim,
            depth=spectral_depth,
            num_heads=spectral_heads,
            dropout=dropout,
            max_bands=max_bands,
        )

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes),
        )

    def forward(self, x):
        spectral_features = self.spectral(x)
        return self.head(spectral_features)

In [13]:
spectral_model = SpectralOnlyClassifier(
    num_classes=len(SALINAS_CLASS_NAMES),
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    dropout=0.1,
    max_bands=256,
).to(device)

print(spectral_model)
print(
    "Trainable parameters:",
    count_parameters(spectral_model)
)

SpectralOnlyClassifier(
  (spectral): SpectralAttentionBranch(
    (value_projection): Linear(in_features=1, out_features=128, bias=True)
    (band_embedding): Embedding(256, 128)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (head): Seq

In [14]:
x, y = next(iter(train_loader))

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = spectral_model(x)

print("Input :", x.shape)
print("Output:", logits.shape)

Input : torch.Size([128, 204, 15, 15])
Output: torch.Size([128, 16])


In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    spectral_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    total_samples = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False,
    )

    for x, y in progress:
        x = x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )
        y = y.to(
            device=device,
            dtype=torch.long,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = x.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / total_samples

In [22]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = macro_f1_score(
        y_true,
        y_pred,
    )

    return accuracy, macro_f1

In [23]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    spectral_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("Training functions ready.")

Training functions ready.


In [24]:
test_loss = train_one_epoch(
    spectral_model,
    train_loader,
    optimizer,
    criterion,
    device,
)

val_acc, val_f1 = evaluate(
    spectral_model,
    val_loader,
    device,
)

print(f"Loss: {test_loss:.4f}")
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val Macro-F1: {val_f1:.4f}")

Loss: 0.5082
Val Accuracy: 0.8220
Val Macro-F1: 0.7090


In [25]:
NUM_EPOCHS = 50

best_spectral_state = None
best_spectral_epoch = None
best_spectral_macro_f1 = -float("inf")

spectral_history = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = train_one_epoch(
        spectral_model,
        train_loader,
        optimizer,
        criterion,
        device,
    )

    val_accuracy, val_macro_f1 = evaluate(
        spectral_model,
        val_loader,
        device,
    )

    spectral_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

    if val_macro_f1 > best_spectral_macro_f1:
        best_spectral_macro_f1 = val_macro_f1
        best_spectral_epoch = epoch

        best_spectral_state = {
            k: v.detach().cpu().clone()
            for k, v in spectral_model.state_dict().items()
        }

        print("✓ New best spectral-only model")

Epoch 01 | Loss: 0.1482 | Val Acc: 0.8424 | Val Macro-F1: 0.7651
✓ New best spectral-only model


Epoch 02 | Loss: 0.0985 | Val Acc: 0.7964 | Val Macro-F1: 0.7710
✓ New best spectral-only model


Epoch 03 | Loss: 0.0643 | Val Acc: 0.9317 | Val Macro-F1: 0.8389
✓ New best spectral-only model


Epoch 04 | Loss: 0.0538 | Val Acc: 0.9367 | Val Macro-F1: 0.8666
✓ New best spectral-only model


Epoch 05 | Loss: 0.0492 | Val Acc: 0.9633 | Val Macro-F1: 0.8826
✓ New best spectral-only model


...

Epoch 46 | Loss: 0.0114 | Val Acc: 0.9178 | Val Macro-F1: 0.8577


Epoch 47 | Loss: 0.0177 | Val Acc: 0.9175 | Val Macro-F1: 0.8695


Epoch 48 | Loss: 0.0164 | Val Acc: 0.9554 | Val Macro-F1: 0.8665


Epoch 49 | Loss: 0.0100 | Val Acc: 0.9123 | Val Macro-F1: 0.8563


Epoch 50 | Loss: 0.0061 | Val Acc: 0.8556 | Val Macro-F1: 0.8232


In [26]:
spectral_history_df = pd.DataFrame(spectral_history)

spectral_history_df.to_csv(
    outputs_dir / "spectral_only_spatial_history.csv",
    index=False,
)

print("Best epoch:", best_spectral_epoch)
print("Best validation Macro-F1:", best_spectral_macro_f1)

Best epoch: 33
Best validation Macro-F1: 0.9168993288471428


In [27]:
spectral_model.load_state_dict(best_spectral_state)
spectral_model.to(device)
spectral_model.eval()

print("Restored best spectral epoch:", best_spectral_epoch)
print("Best validation Macro-F1:", best_spectral_macro_f1)

Restored best spectral epoch: 33
Best validation Macro-F1: 0.9168993288471428


In [29]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Collecting test predictions",
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return y_true, y_pred

In [30]:
y_test_spectral, y_pred_spectral = collect_predictions(
    spectral_model,
    test_loader,
)

print("Test samples:", len(y_test_spectral))
print("Predictions:", len(y_pred_spectral))

Test samples: 10840
Predictions: 10840


In [31]:
spectral_test_accuracy = accuracy_score(
    y_test_spectral,
    y_pred_spectral,
)

spectral_test_macro_f1 = macro_f1_score(
    y_test_spectral,
    y_pred_spectral,
)

print("=" * 55)
print("SPECTRAL-ONLY SPATIAL TEST RESULTS")
print("=" * 55)
print(f"Accuracy : {spectral_test_accuracy:.4f}")
print(f"Macro-F1 : {spectral_test_macro_f1:.4f}")

SPECTRAL-ONLY SPATIAL TEST RESULTS
Accuracy : 0.9520
Macro-F1 : 0.9020


In [32]:
class ConcatHybridClassifier(nn.Module):
    """
    Spatial CNN + Spectral Attention with simple concatenation.
    This removes the gated fusion mechanism while keeping the
    two feature branches unchanged.
    """

    def __init__(
        self,
        in_channels,
        num_classes,
        feature_dim=128,
        spectral_depth=2,
        spectral_heads=4,
        dropout=0.1,
        max_bands=256,
    ):
        super().__init__()

        self.spatial = SpatialCNNBranch(
            in_channels=in_channels,
            feature_dim=feature_dim,
        )

        self.spectral = SpectralAttentionBranch(
            feature_dim=feature_dim,
            depth=spectral_depth,
            num_heads=spectral_heads,
            dropout=dropout,
            max_bands=max_bands,
        )

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, num_classes),
        )

    def forward(self, x):
        spatial_features = self.spatial(x)
        spectral_features = self.spectral(x)

        fused = torch.cat(
            [spatial_features, spectral_features],
            dim=-1,
        )

        return self.head(fused)

In [34]:
from dfm.models.hybrid import (
    SpatialCNNBranch,
    SpectralAttentionBranch,
)

In [35]:
concat_model = ConcatHybridClassifier(
    in_channels=scene.bands,
    num_classes=len(SALINAS_CLASS_NAMES),
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    dropout=0.1,
    max_bands=256,
).to(device)

print(concat_model)
print(
    "Trainable parameters:",
    count_parameters(concat_model)
)

ConcatHybridClassifier(
  (spatial): SpatialCNNBranch(
    (net): Sequential(
      (0): Conv2d(204, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): AdaptiveAvgPool2d(output_size=1)
      (10): Flatten(start_dim=1, end_dim=-1)
    )
  )
  (spectral): SpectralAttentionBranch(
    (value_projection): Linear(in_features=1, out_features=128, bias=True)
    (band_embedding): Embedding(256, 128)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
     

In [36]:
x, y = next(iter(train_loader))

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = concat_model(x)

print("Input :", x.shape)
print("Output:", logits.shape)

Input : torch.Size([128, 204, 15, 15])
Output: torch.Size([128, 16])


In [37]:
criterion_concat = nn.CrossEntropyLoss()

optimizer_concat = torch.optim.AdamW(
    concat_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

In [38]:
NUM_EPOCHS = 50

best_concat_state = None
best_concat_epoch = None
best_concat_macro_f1 = -float("inf")

concat_history = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = train_one_epoch(
        concat_model,
        train_loader,
        optimizer_concat,
        criterion_concat,
        device,
    )

    val_accuracy, val_macro_f1 = evaluate(
        concat_model,
        val_loader,
        device,
    )

    concat_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

    if val_macro_f1 > best_concat_macro_f1:
        best_concat_macro_f1 = val_macro_f1
        best_concat_epoch = epoch

        best_concat_state = {
            k: v.detach().cpu().clone()
            for k, v in concat_model.state_dict().items()
        }

        print("✓ New best concat model")

Epoch 01 | Loss: 0.2637 | Val Acc: 0.7643 | Val Macro-F1: 0.7398
✓ New best concat model


Epoch 02 | Loss: 0.0361 | Val Acc: 0.7840 | Val Macro-F1: 0.7757
✓ New best concat model


Epoch 03 | Loss: 0.0267 | Val Acc: 0.7842 | Val Macro-F1: 0.7498


Epoch 04 | Loss: 0.0166 | Val Acc: 0.8542 | Val Macro-F1: 0.8334
✓ New best concat model


Epoch 05 | Loss: 0.0152 | Val Acc: 0.8468 | Val Macro-F1: 0.7868


...

Epoch 46 | Loss: 0.0017 | Val Acc: 0.7985 | Val Macro-F1: 0.8737


Epoch 47 | Loss: 0.0037 | Val Acc: 0.8571 | Val Macro-F1: 0.9057


Epoch 48 | Loss: 0.0040 | Val Acc: 0.9242 | Val Macro-F1: 0.9485


Epoch 49 | Loss: 0.0014 | Val Acc: 0.8908 | Val Macro-F1: 0.9241


Epoch 50 | Loss: 0.0010 | Val Acc: 0.9344 | Val Macro-F1: 0.9493


In [39]:
concat_history_df = pd.DataFrame(concat_history)

concat_history_df.to_csv(
    outputs_dir / "concat_hybrid_spatial_history.csv",
    index=False,
)

print("Best epoch:", best_concat_epoch)
print(
    "Best validation Macro-F1:",
    best_concat_macro_f1
)

Best epoch: 37
Best validation Macro-F1: 0.9567266628573079


In [40]:
concat_model.load_state_dict(best_concat_state)
concat_model.to(device)
concat_model.eval()

print("Restored best Concat epoch:", best_concat_epoch)
print("Best validation Macro-F1:", best_concat_macro_f1)

Restored best Concat epoch: 37
Best validation Macro-F1: 0.9567266628573079


In [41]:
y_test_concat, y_pred_concat = collect_predictions(
    concat_model,
    test_loader,
)

print("Test samples:", len(y_test_concat))
print("Predictions:", len(y_pred_concat))

Test samples: 10840
Predictions: 10840


In [42]:
concat_test_accuracy = accuracy_score(
    y_test_concat,
    y_pred_concat,
)

concat_test_macro_f1 = macro_f1_score(
    y_test_concat,
    y_pred_concat,
)

print("=" * 55)
print("CONCAT HYBRID SPATIAL TEST RESULTS")
print("=" * 55)
print(f"Accuracy : {concat_test_accuracy:.4f}")
print(f"Macro-F1 : {concat_test_macro_f1:.4f}")

CONCAT HYBRID SPATIAL TEST RESULTS
Accuracy : 0.9744
Macro-F1 : 0.9627


In [43]:
ablation_results = pd.DataFrame([
    {
        "model": "CNN / Spatial-only",
        "accuracy": 0.960148,
        "macro_f1": 0.941450,
    },
    {
        "model": "Spectral-only",
        "accuracy": 0.9520,
        "macro_f1": 0.9020,
    },
    {
        "model": "Concat Hybrid",
        "accuracy": 0.9744,
        "macro_f1": 0.9627,
    },
    {
        "model": "Gated Hybrid",
        "accuracy": 0.980996,
        "macro_f1": 0.963905,
    },
])

ablation_results.to_csv(
    outputs_dir / "hybrid_ablation_comparison.csv",
    index=False,
)

display(ablation_results)

,model,accuracy,macro_f1
0,CNN / Spatial-only,0.960148,0.941450
1,Spectral-only,0.952000,0.902000
2,Concat Hybrid,0.974400,0.962700
3,Gated Hybrid,0.980996,0.963905
